# Problem 1 – RNN, LSTM and GRU on Sequences
**Student:** Ayushi Pradhan &nbsp;|&nbsp; **ID:** 2026AJ05077  
**Variant:** Last digit 7 (Odd) → Task 5(c) ablation: LSTM forget-gate bias 0 vs 1 at T=200  
**Hidden size:** 64 &nbsp;|&nbsp; **Seeds:** 0, 1, 2


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Imports and global seed helper                        ║
# ╚══════════════════════════════════════════════════════════════════╝


import os, random, time, urllib.request, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # non-interactive backend for Colab / scripts
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (accuracy_score, f1_score,
                              confusion_matrix, classification_report)

warnings.filterwarnings('ignore')

# ── Reproducibility helper ───────────────────────────────────────────────────
def set_seed(seed: int) -> None:
    """Fix all relevant random states so runs are reproducible.
    
    PyTorch may still show tiny GPU-kernel differences; that is expected
    and not penalised per the assignment spec.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Deterministic cuDNN (slightly slower, but ensures bit-exact ops where possible)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

# ── Device setup ─────────────────────────────────────────────────────────────
# Prefer CUDA (Colab T4), fall back to CPU gracefully
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# ── Global hyper-parameters (variant-specific values) ────────────────────────
HIDDEN_SIZE  = 64          # from variants table: student last-digit 7
SEEDS        = [0, 1, 2]   # three fixed seeds for all comparisons
N_EPOCHS_HAR = 30          # same budget for every HAR model
N_EPOCHS_ADD = 60          # same budget for adding-problem models
LR           = 1e-3        # Adam learning rate (assignment default)
BATCH_SIZE   = 128         # fixed across all models for fair comparison

# Figure output directory
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)
print("Setup complete.")


Using device: cpu
Setup complete.


## Task 1 – Data Pipelines

In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Download & unzip UCI HAR dataset                      ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# UCI HAR: 30 subjects, 128-step windows of 9 inertial channels,
# 6 activity classes (walking=1 .. laying=6).
# We download once and cache locally.

HAR_URL   = ('https://archive.ics.uci.edu/static/public/240/'
             'human+activity+recognition+using+smartphones.zip')
HAR_ZIP   = 'har.zip'
HAR_DIR   = 'har'
INNER_ZIP = os.path.join(HAR_DIR, 'UCI HAR Dataset.zip')   # nested zip

if not os.path.isdir(os.path.join(HAR_DIR, 'UCI HAR Dataset')):
    print("Downloading UCI HAR (~58 MB)…")
    urllib.request.urlretrieve(HAR_URL, HAR_ZIP)
    print("Unzipping outer archive…")
    with zipfile.ZipFile(HAR_ZIP) as z:
        z.extractall(HAR_DIR)
    # The outer zip sometimes contains a second inner zip
    if os.path.isfile(INNER_ZIP):
        print("Unzipping inner archive…")
        with zipfile.ZipFile(INNER_ZIP) as z:
            z.extractall(HAR_DIR)
    print("Done.")
else:
    print("UCI HAR already present, skipping download.")

HAR_ROOT = os.path.join(HAR_DIR, 'UCI HAR Dataset')
print(f"HAR root: {HAR_ROOT}")


ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Load inertial signals and build tensors               ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Channel order (fixed for all students):
#   0  body_acc_x    3  body_gyro_x    6  total_acc_x
#   1  body_acc_y    4  body_gyro_y    7  total_acc_y
#   2  body_acc_z    5  body_gyro_z    8  total_acc_z
#
# Each signal file: (n_samples, 128) – one row per 2-second window.

CHANNELS = [
    'body_acc_x',  'body_acc_y',  'body_acc_z',
    'body_gyro_x', 'body_gyro_y', 'body_gyro_z',
    'total_acc_x', 'total_acc_y', 'total_acc_z',
]

def load_har_split(split: str):
    """Return (X, y, subjects) for 'train' or 'test'.

    X shape: (N, 128, 9)   – float32
    y shape: (N,)          – int64, 0-based (original labels are 1-based)
    subjects: (N,)         – subject IDs
    """
    sig_dir = os.path.join(HAR_ROOT, split, 'Inertial Signals')
    channels = []
    for ch in CHANNELS:
        # e.g. 'body_acc_x_train.txt'
        fpath = os.path.join(sig_dir, f'{ch}_{split}.txt')
        # Each row is 128 whitespace-separated floats
        arr = np.loadtxt(fpath, dtype=np.float32)  # (N, 128)
        channels.append(arr)
    # Stack along new last axis → (N, 128, 9)
    X = np.stack(channels, axis=-1)

    y_path  = os.path.join(HAR_ROOT, split, f'y_{split}.txt')
    s_path  = os.path.join(HAR_ROOT, split, f'subject_{split}.txt')
    y       = np.loadtxt(y_path,  dtype=np.int64) - 1   # 0-based
    subjects = np.loadtxt(s_path, dtype=np.int64)
    return X, y, subjects

X_train_full, y_train_full, subj_train = load_har_split('train')
X_test,       y_test,       subj_test  = load_har_split('test')

print(f"Train full  : X={X_train_full.shape}, y={y_train_full.shape}")
print(f"Test        : X={X_test.shape},       y={y_test.shape}")
# Expected: Train (7352,128,9), Test (2947,128,9)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Subject-wise validation split                         ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Assignment rule: hold out 4 of the 21 training subjects as a
# validation set. The held-out subjects are chosen via the given
# formula using each seed in turn. Here we use SEED=0 for the
# canonical split; seeds 1 and 2 are used during the 3-seed training
# loop (Cell 7) to show variance.
#
# IMPORTANT: normalisation statistics (mean, std per channel) are
# computed ONLY on the training windows (not validation, not test).

train_subjects = np.unique(subj_train)   # 21 unique subject IDs
print(f"Training subjects ({len(train_subjects)}): {train_subjects}")

def make_val_split(X_full, y_full, subjects_full, seed):
    """Hold out 4 subjects from training for validation.
    
    Returns (X_tr, y_tr, X_val, y_val) with per-channel normalisation
    fitted on the training portion only.
    """
    rng  = np.random.default_rng(seed)
    unique_subjs = np.unique(subjects_full)
    held_out     = rng.choice(unique_subjs, 4, replace=False)

    val_mask  = np.isin(subjects_full, held_out)
    train_mask = ~val_mask

    X_tr  = X_full[train_mask]   # (N_tr, 128, 9)
    y_tr  = y_full[train_mask]
    X_val = X_full[val_mask]     # (N_val, 128, 9)
    y_val = y_full[val_mask]

    # Per-channel mean & std over all windows AND timesteps in training set
    # shape of X_tr: (N, T, C) → compute stats over axes 0 and 1
    mean = X_tr.mean(axis=(0, 1), keepdims=True)   # (1,1,9)
    std  = X_tr.std(axis=(0, 1),  keepdims=True) + 1e-8

    X_tr  = (X_tr  - mean) / std
    X_val = (X_val - mean) / std   # use TRAINING statistics for val too

    print(f"  Seed {seed}: train={X_tr.shape[0]}, val={X_val.shape[0]}, "
          f"held-out subjects={sorted(held_out)}")
    return X_tr, y_tr, X_val, y_val, mean, std

# Canonical split for single-run experiments
X_tr, y_tr, X_val, y_val, har_mean, har_std = make_val_split(
    X_train_full, y_train_full, subj_train, seed=0)

# Apply same training-set statistics to test set
X_test_norm = (X_test - har_mean) / har_std
print(f"Normalised test: {X_test_norm.shape}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Figure 1: Class-balance bar chart                     ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# A well-balanced dataset → we can use accuracy as a primary metric
# (macro-F1 is still reported to catch any per-class skew).

ACTIVITY_LABELS = [
    'Walking', 'Walk Up', 'Walk Down',
    'Sitting', 'Standing', 'Laying'
]

counts_train = np.bincount(y_tr,       minlength=6)
counts_val   = np.bincount(y_val,      minlength=6)
counts_test  = np.bincount(y_test,     minlength=6)

x     = np.arange(6)
width = 0.28

fig, ax = plt.subplots(figsize=(9, 4))
bars_tr  = ax.bar(x - width, counts_train, width, label='Train',      color='steelblue',   alpha=0.85)
bars_val = ax.bar(x,          counts_val,  width, label='Validation', color='darkorange',  alpha=0.85)
bars_te  = ax.bar(x + width, counts_test,  width, label='Test',       color='seagreen',    alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(ACTIVITY_LABELS, fontsize=9)
ax.set_ylabel('Window count')
ax.set_title('Figure 1: UCI HAR class distribution across splits')
ax.legend()
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

# Annotate bar tops
for bars in [bars_tr, bars_val, bars_te]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 15,
                str(h), ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig1_class_balance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 1 saved.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Adding-problem generator and sanity tests (Task 1)    ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Each sample has shape (T, 2):
#   channel 0 : random uniform value in [0, 1)
#   channel 1 : binary marker (exactly two 1s per sequence)
# Target : sum of the two channel-0 values at the marked positions.
# Baseline MSE : a model that always predicts 1.0 → MSE ≈ 0.167.

def make_adding(n: int, T: int, seed: int):
    """Generate the adding-problem dataset as specified in the assignment.

    Parameters
    ----------
    n    : number of sequences
    T    : sequence length
    seed : RNG seed for reproducibility

    Returns
    -------
    x : np.ndarray, shape (n, T, 2), float32
    y : np.ndarray, shape (n,),      float32
    """
    rng  = np.random.default_rng(seed)
    x    = np.zeros((n, T, 2), dtype=np.float32)
    x[:, :, 0] = rng.random((n, T))                 # channel 0: random values
    i1   = rng.integers(0,     T // 2, n)            # first  marker index (first half)
    i2   = rng.integers(T // 2, T,    n)             # second marker index (second half)
    x[np.arange(n), i1, 1] = 1                      # set marker channel
    x[np.arange(n), i2, 1] = 1
    y    = x[np.arange(n), i1, 0] + x[np.arange(n), i2, 0]
    return x, y.astype(np.float32)

# ── Unit tests (assert cells) ────────────────────────────────────────────────
# Test 1: exactly two markers per sequence
x_test_add, y_test_add = make_adding(500, 100, seed=42)
marker_counts = x_test_add[:, :, 1].sum(axis=1)   # (500,) – should all equal 2
assert np.all(marker_counts == 2), "FAIL: not exactly 2 markers per sequence"
print("✓ Test 1 passed: every sequence has exactly 2 markers")

# Test 2: one marker in first half, one in second half
markers_per_seq = np.where(x_test_add[:, :, 1] == 1)   # row indices, col indices
# Group positions by sequence
for i in range(min(500, x_test_add.shape[0])):
    positions = np.where(x_test_add[i, :, 1] == 1)[0]
    assert len(positions) == 2
    assert positions[0] < 50  , f"Seq {i}: first marker not in first half"
    assert positions[1] >= 50 , f"Seq {i}: second marker not in second half"
print("✓ Test 2 passed: markers correctly placed in each half")

# Test 3: target equals sum of marked values
for i in range(min(500, x_test_add.shape[0])):
    positions  = np.where(x_test_add[i, :, 1] == 1)[0]
    expected   = x_test_add[i, positions[0], 0] + x_test_add[i, positions[1], 0]
    assert abs(y_test_add[i] - expected) < 1e-6, f"Seq {i}: target mismatch"
print("✓ Test 3 passed: target = sum of marked values")

# Baseline MSE (always predict 1.0)
baseline_mse = np.mean((y_test_add - 1.0)**2)
print(f"\nBaseline MSE (predict 1.0): {baseline_mse:.4f}  (expected ≈ 0.167)")


## Task 2 – Recurrent Cells from Scratch

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — MyRNNCell: vanilla RNN from equations                 ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Equation:  h_t = tanh( x_t W_ih^T + b_ih  +  h_{t-1} W_hh^T + b_hh )
#
# We implement this with plain matrix multiplications (no nn.RNN inside).
# After building the cell we copy weights from nn.RNNCell to verify
# numerical equivalence within 1e-5.

class MyRNNCell(nn.Module):
    """Vanilla RNN cell implemented from scratch using matrix ops.

    Parameters
    ----------
    input_size  : dimension of the input vector  d_in
    hidden_size : dimension of the hidden state   h
    """
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.hidden_size = hidden_size
        # Input-to-hidden weight matrix: W_ih ∈ R^{h × d_in}
        self.W_ih = nn.Parameter(torch.empty(hidden_size, input_size))
        # Hidden-to-hidden weight matrix: W_hh ∈ R^{h × h}
        self.W_hh = nn.Parameter(torch.empty(hidden_size, hidden_size))
        # Bias vectors
        self.b_ih = nn.Parameter(torch.zeros(hidden_size))
        self.b_hh = nn.Parameter(torch.zeros(hidden_size))
        # Initialise weights with Kaiming uniform (same as PyTorch default)
        nn.init.kaiming_uniform_(self.W_ih, a=0)
        nn.init.kaiming_uniform_(self.W_hh, a=0)

    def forward(self, x: torch.Tensor, h: torch.Tensor) -> torch.Tensor:
        """One step of the vanilla RNN.

        x : (B, d_in)   – current input
        h : (B, h)      – previous hidden state

        Returns
        -------
        h_new : (B, h)  – next hidden state
        """
        # x @ W_ih.T  →  (B, h)   (contribution from input)
        # h @ W_hh.T  →  (B, h)   (contribution from previous hidden state)
        h_new = torch.tanh(
            x @ self.W_ih.t() + self.b_ih +
            h @ self.W_hh.t() + self.b_hh
        )
        return h_new


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — MyLSTMCell: LSTM from equations                       ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# LSTM equations (input i, forget f, cell gate g, output o):
#
#   [ i ]                             [ x_t  ]
#   [ f ] = σ / σ / tanh / σ  of  W_ih|------| + b_ih + W_hh h_{t-1} + b_hh
#   [ g ]                             [ h_{t-1} ]
#   [ o ]
#
# c_t   = f ⊙ c_{t-1}  +  i ⊙ g
# h_t   = o ⊙ tanh(c_t)
#
# W_ih ∈ R^{4h × d_in},  W_hh ∈ R^{4h × h}  → 4 gate blocks concatenated

class MyLSTMCell(nn.Module):
    """LSTM cell implemented from scratch.

    The four gate weight matrices (i, f, g, o) are stored as a single
    concatenated parameter (matching PyTorch convention) so weight copying
    from nn.LSTMCell is trivial.
    """
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.hidden_size = hidden_size
        # Concatenated weight for 4 gates: shape (4h, d_in)
        self.W_ih = nn.Parameter(torch.empty(4 * hidden_size, input_size))
        # Concatenated recurrent weight: shape (4h, h)
        self.W_hh = nn.Parameter(torch.empty(4 * hidden_size, hidden_size))
        # Bias terms (PyTorch keeps two separate biases b_ih and b_hh)
        self.b_ih = nn.Parameter(torch.zeros(4 * hidden_size))
        self.b_hh = nn.Parameter(torch.zeros(4 * hidden_size))
        nn.init.kaiming_uniform_(self.W_ih, a=0)
        nn.init.kaiming_uniform_(self.W_hh, a=0)

    def forward(self, x: torch.Tensor,
                state: tuple) -> tuple:
        """One LSTM step.

        x     : (B, d_in)
        state : (h, c)  each (B, h)

        Returns (h_new, c_new) each (B, h)
        """
        h, c = state
        h_size = self.hidden_size

        # Combined pre-activation: (B, 4h)
        # x  @ W_ih.T → (B, 4h)   input contribution
        # h  @ W_hh.T → (B, 4h)   recurrent contribution
        gates = x @ self.W_ih.t() + self.b_ih + h @ self.W_hh.t() + self.b_hh

        # Split into four gate vectors, each (B, h)
        i = torch.sigmoid(gates[:, 0*h_size : 1*h_size])    # input gate
        f = torch.sigmoid(gates[:, 1*h_size : 2*h_size])    # forget gate
        g = torch.tanh   (gates[:, 2*h_size : 3*h_size])    # cell gate
        o = torch.sigmoid(gates[:, 3*h_size : 4*h_size])    # output gate

        c_new = f * c + i * g                 # cell state update
        h_new = o * torch.tanh(c_new)         # hidden state update
        return h_new, c_new


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — MyGRUCell: GRU from equations                        ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# GRU equations (reset r, update z, new gate n):
#
#   r   = σ( W_ir x_t + b_ir  +  W_hr h_{t-1} + b_hr )
#   z   = σ( W_iz x_t + b_iz  +  W_hz h_{t-1} + b_hz )
#   n   = tanh( W_in x_t + b_in  +  r ⊙ (W_hn h_{t-1} + b_hn) )
#   h_t = (1 − z) ⊙ n  +  z ⊙ h_{t-1}
#
# W_ih ∈ R^{3h × d_in}  (gates [r, z, n] stacked)
# W_hh ∈ R^{3h × h}

class MyGRUCell(nn.Module):
    """GRU cell implemented from scratch."""
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.hidden_size = hidden_size
        # Input weight for [r, z, n]: (3h, d_in)
        self.W_ih = nn.Parameter(torch.empty(3 * hidden_size, input_size))
        # Recurrent weight for [r, z, n]: (3h, h)
        self.W_hh = nn.Parameter(torch.empty(3 * hidden_size, hidden_size))
        self.b_ih = nn.Parameter(torch.zeros(3 * hidden_size))
        self.b_hh = nn.Parameter(torch.zeros(3 * hidden_size))
        nn.init.kaiming_uniform_(self.W_ih, a=0)
        nn.init.kaiming_uniform_(self.W_hh, a=0)

    def forward(self, x: torch.Tensor, h: torch.Tensor) -> torch.Tensor:
        """One GRU step.

        x : (B, d_in)
        h : (B, h)

        Returns h_new : (B, h)
        """
        h_size = self.hidden_size
        # Input contribution: (B, 3h)
        x_gates = x @ self.W_ih.t() + self.b_ih
        # Recurrent contribution: (B, 3h)
        h_gates = h @ self.W_hh.t() + self.b_hh

        # Reset and update gates use full combined signal
        r = torch.sigmoid(x_gates[:, 0*h_size:1*h_size] +
                          h_gates[:, 0*h_size:1*h_size])   # reset gate
        z = torch.sigmoid(x_gates[:, 1*h_size:2*h_size] +
                          h_gates[:, 1*h_size:2*h_size])   # update gate

        # Candidate hidden: reset gate modulates the recurrent term only
        n = torch.tanh(
            x_gates[:, 2*h_size:3*h_size] +
            r * h_gates[:, 2*h_size:3*h_size]
        )
        h_new = (1 - z) * n + z * h                        # output blending
        return h_new


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Verify cells match library within 1e-5               ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Parameter counts (d=9, h=64):
#   RNN  : W_ih(64×9) + W_hh(64×64) = 576 + 4096 = 4,672
#          + b_ih(64) + b_hh(64)     → MY count  = 4,800  ← 2 biases
#          PyTorch reports 4,800  (two bias vectors b_ih and b_hh)
#   LSTM : W_ih(256×9) + W_hh(256×64) = 2,304 + 16,384 = 18,688
#          + b_ih(256) + b_hh(256)      → MY count = 19,200
#          PyTorch reports 19,200 (two bias vectors)
#   GRU  : W_ih(192×9) + W_hh(192×64) = 1,728 + 12,288 = 14,016
#          + b_ih(192) + b_hh(192)      → MY count = 14,400
#          PyTorch reports 14,400 (two bias vectors)
#
# Note: the assignment quote (4,736 / 18,944 / 14,208) is the count
# with ONE bias only; PyTorch uses TWO → adds h (or 4h/3h) extra params.

set_seed(0)
D_IN  = 9
H     = 64
BATCH = 16
T     = 10

torch.manual_seed(0)
x_seq = torch.randn(BATCH, T, D_IN)   # (B, T, d_in)
h0    = torch.zeros(BATCH, H)
c0    = torch.zeros(BATCH, H)

# ── RNN verification ─────────────────────────────────────────────────────────
lib_rnn  = nn.RNNCell(D_IN, H)
my_rnn   = MyRNNCell(D_IN, H)
# Copy weights from library → custom cell
with torch.no_grad():
    my_rnn.W_ih.copy_(lib_rnn.weight_ih)
    my_rnn.W_hh.copy_(lib_rnn.weight_hh)
    my_rnn.b_ih.copy_(lib_rnn.bias_ih)
    my_rnn.b_hh.copy_(lib_rnn.bias_hh)

h_lib, h_my = h0.clone(), h0.clone()
for t in range(T):
    h_lib = lib_rnn(x_seq[:, t], h_lib)
    h_my  = my_rnn(x_seq[:, t],  h_my)
rnn_err = (h_lib - h_my).abs().max().item()
assert rnn_err < 1e-5, f"RNN mismatch: {rnn_err}"
print(f"✓ RNN  cell matches nn.RNNCell  within 1e-5  (max diff = {rnn_err:.2e})")
print(f"  Parameter count: {sum(p.numel() for p in my_rnn.parameters())}  "
      f"(library: {sum(p.numel() for p in lib_rnn.parameters())})")

# ── LSTM verification ─────────────────────────────────────────────────────────
lib_lstm = nn.LSTMCell(D_IN, H)
my_lstm  = MyLSTMCell(D_IN, H)
with torch.no_grad():
    my_lstm.W_ih.copy_(lib_lstm.weight_ih)
    my_lstm.W_hh.copy_(lib_lstm.weight_hh)
    my_lstm.b_ih.copy_(lib_lstm.bias_ih)
    my_lstm.b_hh.copy_(lib_lstm.bias_hh)

h_lib, c_lib = h0.clone(), c0.clone()
h_my,  c_my  = h0.clone(), c0.clone()
for t in range(T):
    h_lib, c_lib = lib_lstm(x_seq[:, t], (h_lib, c_lib))
    h_my,  c_my  = my_lstm( x_seq[:, t], (h_my,  c_my))
lstm_err = (h_lib - h_my).abs().max().item()
assert lstm_err < 1e-5, f"LSTM mismatch: {lstm_err}"
print(f"✓ LSTM cell matches nn.LSTMCell within 1e-5  (max diff = {lstm_err:.2e})")
print(f"  Parameter count: {sum(p.numel() for p in my_lstm.parameters())}  "
      f"(library: {sum(p.numel() for p in lib_lstm.parameters())})")

# ── GRU verification ─────────────────────────────────────────────────────────
lib_gru = nn.GRUCell(D_IN, H)
my_gru  = MyGRUCell(D_IN, H)
with torch.no_grad():
    my_gru.W_ih.copy_(lib_gru.weight_ih)
    my_gru.W_hh.copy_(lib_gru.weight_hh)
    my_gru.b_ih.copy_(lib_gru.bias_ih)
    my_gru.b_hh.copy_(lib_gru.bias_hh)

h_lib, h_my = h0.clone(), h0.clone()
for t in range(T):
    h_lib = lib_gru(x_seq[:, t], h_lib)
    h_my  = my_gru(x_seq[:, t],  h_my)
gru_err = (h_lib - h_my).abs().max().item()
assert gru_err < 1e-5, f"GRU mismatch: {gru_err}"
print(f"✓ GRU  cell matches nn.GRUCell  within 1e-5  (max diff = {gru_err:.2e})")
print(f"  Parameter count: {sum(p.numel() for p in my_gru.parameters())}  "
      f"(library: {sum(p.numel() for p in lib_gru.parameters())})")

print("\nAll from-scratch cell verification tests passed ✓")


## Task 3 – HAR Classification

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 11 — HAR classifier model definitions                     ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# All three classifiers share the same architecture:
#   • One recurrent layer (RNN / LSTM / GRU) with hidden_size=64
#   • A linear head projecting the final hidden state to 6 classes
# This ensures the comparison is controlled (same capacity, same head).

class HARClassifier(nn.Module):
    """Recurrent classifier for HAR.

    Takes the full 128-step, 9-channel input and classifies the
    activity from the last hidden state.

    Parameters
    ----------
    rnn_type    : 'RNN' | 'LSTM' | 'GRU'
    input_size  : 9  (number of sensor channels)
    hidden_size : 64 (from student variant)
    n_classes   : 6
    n_layers    : 1
    dropout     : 0.3 (applied between stacked layers, ignored for n_layers=1)
    """
    def __init__(self, rnn_type: str = 'LSTM',
                 input_size: int = 9,
                 hidden_size: int = 64,
                 n_classes: int = 6,
                 n_layers: int = 1,
                 dropout: float = 0.3):
        super().__init__()
        self.rnn_type    = rnn_type
        self.hidden_size = hidden_size
        self.n_layers    = n_layers

        rnn_kwargs = dict(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,          # (B, T, d_in) → (B, T, h)
            dropout=dropout if n_layers > 1 else 0.0,
        )
        if rnn_type == 'RNN':
            self.rnn = nn.RNN(**rnn_kwargs)
        elif rnn_type == 'LSTM':
            self.rnn = nn.LSTM(**rnn_kwargs)
        elif rnn_type == 'GRU':
            self.rnn = nn.GRU(**rnn_kwargs)
        else:
            raise ValueError(f"Unknown rnn_type: {rnn_type}")

        self.dropout = nn.Dropout(p=dropout)
        # Classification head: hidden state → 6 logits
        self.head = nn.Linear(hidden_size, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass.

        x : (B, 128, 9)

        Returns logits : (B, 6)
        """
        # out : (B, T, h),  hidden : depends on cell type
        out, hidden = self.rnn(x)

        # We classify from the LAST hidden state (at step T-1)
        # out[:, -1, :] is equivalent and works for all three types
        last_hidden = out[:, -1, :]     # (B, h)
        last_hidden = self.dropout(last_hidden)
        logits = self.head(last_hidden) # (B, 6)
        return logits

def count_params(model: nn.Module) -> int:
    """Count trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Quick sanity check: build each model, verify output shape, count params
for rnn_type in ['RNN', 'LSTM', 'GRU']:
    m = HARClassifier(rnn_type=rnn_type, hidden_size=HIDDEN_SIZE).to(DEVICE)
    dummy = torch.randn(4, 128, 9).to(DEVICE)
    out   = m(dummy)
    # out should be (4, 6)
    assert out.shape == (4, 6), f"{rnn_type}: unexpected shape {out.shape}"
    print(f"{rnn_type:4s} | params: {count_params(m):,} | output shape: {out.shape}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 12 — Train / evaluate helper and HAR training loop        ║
# ╚══════════════════════════════════════════════════════════════════╝

def train_har(rnn_type: str, seed: int,
              n_epochs: int = N_EPOCHS_HAR,
              batch_size: int = BATCH_SIZE,
              lr: float = LR,
              hidden_size: int = HIDDEN_SIZE):
    """Train a HARClassifier for one seed and return metrics.

    Returns
    -------
    dict with keys: test_acc, macro_f1, params, sec_per_epoch,
                    train_losses, val_losses, preds (for best model)
    """
    set_seed(seed)

    # Subject-wise split using this seed
    X_tr, y_tr, X_val, y_val, mean_, std_ = make_val_split(
        X_train_full, y_train_full, subj_train, seed=seed)
    X_te_norm = (X_test - mean_) / std_

    # Build DataLoaders
    # Tensors: X (float32), y (long)
    def make_loader(X, y, shuffle):
        ds = TensorDataset(torch.tensor(X), torch.tensor(y, dtype=torch.long))
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

    train_loader = make_loader(X_tr,       y_tr,   shuffle=True)
    val_loader   = make_loader(X_val,      y_val,  shuffle=False)
    test_loader  = make_loader(X_te_norm,  y_test, shuffle=False)

    model     = HARClassifier(rnn_type=rnn_type, hidden_size=hidden_size).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    best_val_loss    = float('inf')
    best_state       = None
    train_losses, val_losses = [], []
    total_epoch_time = 0.0

    for epoch in range(n_epochs):
        # ── Training phase ───────────────────────────────────────────────────
        model.train()
        t0 = time.time()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            logits = model(X_batch)       # (B, 6)
            loss   = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
        epoch_time = time.time() - t0
        total_epoch_time += epoch_time

        # ── Validation phase ─────────────────────────────────────────────────
        model.eval()
        v_losses = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                logits = model(X_batch)
                v_losses.append(criterion(logits, y_batch).item())
        val_loss = np.mean(v_losses)

        # Track epoch-level losses for learning curves
        with torch.no_grad():
            t_losses = []
            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                t_losses.append(criterion(model(X_batch), y_batch).item())
        train_losses.append(np.mean(t_losses))
        val_losses.append(val_loss)

        # Save best model by validation loss
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}

    # ── Evaluate best model on test set ─────────────────────────────────────
    model.load_state_dict(best_state)
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            logits     = model(X_batch.to(DEVICE))
            preds      = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_true.extend(y_batch.numpy())

    acc      = accuracy_score(all_true, all_preds)
    macro_f1 = f1_score(all_true, all_preds, average='macro')

    return {
        'test_acc':       acc,
        'macro_f1':       macro_f1,
        'params':         count_params(model),
        'sec_per_epoch':  total_epoch_time / n_epochs,
        'train_losses':   train_losses,
        'val_losses':     val_losses,
        'preds':          all_preds,
        'true':           all_true,
    }

# ── Run HAR comparison: 3 models × 3 seeds ────────────────────────────────────
print("Training HAR classifiers (3 models × 3 seeds)…")
print("This takes a few minutes on a T4 GPU.\n")

har_results = {}   # {rnn_type: list of result dicts, one per seed}
for rnn_type in ['RNN', 'LSTM', 'GRU']:
    har_results[rnn_type] = []
    for seed in SEEDS:
        print(f"  {rnn_type} seed={seed} …", end='', flush=True)
        res = train_har(rnn_type, seed)
        har_results[rnn_type].append(res)
        print(f" acc={res['test_acc']:.4f}  F1={res['macro_f1']:.4f}  "
              f"t/ep={res['sec_per_epoch']:.1f}s")

print("\nHAR training complete.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 13 — Table 1: HAR results summary                         ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# We display mean ± std over the 3 seeds for each model, plus
# parameter count and time per epoch.

rows = []
for rnn_type in ['RNN', 'LSTM', 'GRU']:
    accs     = [r['test_acc']      for r in har_results[rnn_type]]
    f1s      = [r['macro_f1']      for r in har_results[rnn_type]]
    times    = [r['sec_per_epoch'] for r in har_results[rnn_type]]
    params   = har_results[rnn_type][0]['params']   # same for all seeds

    rows.append({
        'Model':         rnn_type,
        'Params':        params,
        'Acc (mean)':    f"{np.mean(accs):.4f}",
        'Acc (±std)':    f"{np.std(accs):.4f}",
        'F1 (mean)':     f"{np.mean(f1s):.4f}",
        'F1 (±std)':     f"{np.std(f1s):.4f}",
        'Sec/epoch':     f"{np.mean(times):.2f}",
    })

har_table = pd.DataFrame(rows)
print("Table 1: HAR Classification Results (mean ± std over 3 seeds)")
print(har_table.to_string(index=False))


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 14 — Figure 2: Confusion matrix of best HAR model         ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# 'Best' = highest test accuracy across all 9 runs.

best_acc, best_res, best_model_name = 0, None, ''
for rnn_type in ['RNN', 'LSTM', 'GRU']:
    for res in har_results[rnn_type]:
        if res['test_acc'] > best_acc:
            best_acc        = res['test_acc']
            best_res        = res
            best_model_name = rnn_type

cm = confusion_matrix(best_res['true'], best_res['preds'])

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=ACTIVITY_LABELS,
            yticklabels=ACTIVITY_LABELS,
            linewidths=0.5, ax=ax)
ax.set_xlabel('Predicted label', fontsize=10)
ax.set_ylabel('True label',      fontsize=10)
ax.set_title(
    f'Figure 2: Confusion matrix – {best_model_name} '
    f'(test acc = {best_acc:.4f})',
    fontsize=11
)
plt.xticks(rotation=30, ha='right', fontsize=8)
plt.yticks(rotation=0,  fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Best model: {best_model_name}  |  Test accuracy: {best_acc:.4f}")
print("Figure 2 saved.")


## Task 4 – Adding Problem

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 15 — Adding-problem model and training loop               ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# The adding problem is a regression task (predict a scalar sum).
# Architecture: single recurrent layer → linear head to scalar.
# Loss: Mean Squared Error (MSE).

class AddingModel(nn.Module):
    """Recurrent regression model for the adding problem.

    Reads T steps of a 2-channel sequence and predicts a scalar.
    """
    def __init__(self, rnn_type: str = 'LSTM',
                 hidden_size: int = HIDDEN_SIZE):
        super().__init__()
        self.rnn_type = rnn_type
        rnn_cls = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}[rnn_type]
        self.rnn  = rnn_cls(input_size=2, hidden_size=hidden_size,
                            batch_first=True)
        self.head = nn.Linear(hidden_size, 1)   # scalar output

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x : (B, T, 2)  →  scalar : (B,)"""
        out, _ = self.rnn(x)
        # Classify from the last time step
        return self.head(out[:, -1, :]).squeeze(-1)   # (B,)


def train_adding(rnn_type: str, T: int, seed: int,
                 n_epochs: int = N_EPOCHS_ADD,
                 batch_size: int = 128,
                 lr: float = LR):
    """Train and evaluate one adding-problem run.

    Returns dict with test_mse, sec_per_epoch, train_mses, val_mses.
    """
    set_seed(seed)

    # Generate datasets with fixed seeds (per assignment spec)
    x_tr,  y_tr  = make_adding(10000, T, seed=0)   # train
    x_val, y_val = make_adding( 1000, T, seed=1)   # validation
    x_te,  y_te  = make_adding( 1000, T, seed=2)   # test

    def make_loader(x, y, shuffle):
        ds = TensorDataset(torch.tensor(x), torch.tensor(y))
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

    train_loader = make_loader(x_tr,  y_tr,  True)
    val_loader   = make_loader(x_val, y_val, False)
    test_loader  = make_loader(x_te,  y_te,  False)

    model     = AddingModel(rnn_type=rnn_type).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    train_mses, val_mses    = [], []
    total_epoch_time        = 0.0

    for epoch in range(n_epochs):
        model.train()
        t0 = time.time()
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            optimizer.zero_grad()
            pred = model(X_b)
            loss = criterion(pred, y_b)
            loss.backward()
            optimizer.step()
        total_epoch_time += time.time() - t0

        # Epoch losses for learning curves
        model.eval()
        with torch.no_grad():
            t_mse = np.mean([criterion(model(X.to(DEVICE)), y.to(DEVICE)).item()
                             for X, y in train_loader])
            v_mse = np.mean([criterion(model(X.to(DEVICE)), y.to(DEVICE)).item()
                             for X, y in val_loader])
        train_mses.append(t_mse)
        val_mses.append(v_mse)

    # Test MSE
    model.eval()
    test_mse_val = np.mean([
        criterion(model(X.to(DEVICE)), y.to(DEVICE)).item()
        for X, y in test_loader
    ])

    return {
        'test_mse':      test_mse_val,
        'sec_per_epoch': total_epoch_time / n_epochs,
        'train_mses':    train_mses,
        'val_mses':      val_mses,
    }

# ── Run: 4 lengths × 3 models × 3 seeds ──────────────────────────────────────
T_VALUES = [50, 100, 200, 400]
print("Training adding-problem models…")
adding_results = {}  # {(rnn_type, T): [res_seed0, res_seed1, res_seed2]}

for T_val in T_VALUES:
    for rnn_type in ['RNN', 'LSTM', 'GRU']:
        key = (rnn_type, T_val)
        adding_results[key] = []
        for seed in SEEDS:
            print(f"  {rnn_type} T={T_val:3d} seed={seed} …", end='', flush=True)
            res = train_adding(rnn_type, T_val, seed)
            adding_results[key].append(res)
            solved = "✓ SOLVED" if res['test_mse'] < 0.05 else "✗"
            print(f" MSE={res['test_mse']:.4f} {solved}")

print("\nAdding-problem training complete.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 16 — Table 2 & Figures 3-4: Adding-problem results        ║
# ╚══════════════════════════════════════════════════════════════════╝

BASELINE_MSE = 0.167   # always-predict-1.0 baseline

# ── Table 2: MSE summary ─────────────────────────────────────────────────────
rows = []
for T_val in T_VALUES:
    for rnn_type in ['RNN', 'LSTM', 'GRU']:
        mses   = [r['test_mse'] for r in adding_results[(rnn_type, T_val)]]
        solved = sum(1 for m in mses if m < 0.05)
        rows.append({
            'T':           T_val,
            'Model':       rnn_type,
            'MSE (mean)':  f"{np.mean(mses):.4f}",
            'MSE (±std)':  f"{np.std(mses):.4f}",
            'Solved /3':   f"{solved}/3",
        })

adding_table = pd.DataFrame(rows)
print("Table 2: Adding-Problem MSE (mean ± std over 3 seeds)")
print(adding_table.to_string(index=False))
print(f"Baseline MSE (always predict 1.0): {BASELINE_MSE}")

# ── Figure 3: Learning curves at T=200 ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
colors    = {'RNN': 'tab:red', 'LSTM': 'tab:blue', 'GRU': 'tab:green'}

for ax, rnn_type in zip(axes, ['RNN', 'LSTM', 'GRU']):
    for seed_idx, seed in enumerate(SEEDS):
        res = adding_results[(rnn_type, 200)][seed_idx]
        ep  = range(1, len(res['train_mses']) + 1)
        ax.semilogy(ep, res['train_mses'],
                    color=colors[rnn_type], alpha=0.4, lw=1,
                    label='Train' if seed_idx == 0 else '')
        ax.semilogy(ep, res['val_mses'],
                    color=colors[rnn_type], alpha=0.9, lw=1.5, ls='--',
                    label='Val' if seed_idx == 0 else '')
    ax.axhline(BASELINE_MSE, color='grey', ls=':', lw=1.5, label='Baseline 0.167')
    ax.set_title(rnn_type, fontsize=12)
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=8)
    ax.set_ylim(bottom=1e-3)

axes[0].set_ylabel('MSE (log scale)')
fig.suptitle('Figure 3: Adding-problem learning curves at T=200', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_adding_curves_T200.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 4: MSE vs T (mean over seeds) ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
for rnn_type in ['RNN', 'LSTM', 'GRU']:
    means  = [np.mean([r['test_mse'] for r in adding_results[(rnn_type, T_val)]])
              for T_val in T_VALUES]
    stds   = [np.std( [r['test_mse'] for r in adding_results[(rnn_type, T_val)]])
              for T_val in T_VALUES]
    ax.errorbar(T_VALUES, means, yerr=stds,
                label=rnn_type, marker='o', capsize=4,
                color=colors[rnn_type])

ax.axhline(BASELINE_MSE, color='grey', ls=':', lw=1.5, label='Baseline 0.167')
ax.set_xlabel('Sequence length T')
ax.set_ylabel('Test MSE')
ax.set_title('Figure 4: Test MSE vs sequence length T')
ax.legend()
ax.set_xticks(T_VALUES)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig4_mse_vs_T.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figures 3 & 4 saved.")


## Task 5 – Gradients, Ablations and Analysis

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 17 — Figure 5: Gradient norm vs time step (Task 5a)       ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# We track ‖∂L/∂h_t‖ at each step t for T=100 by computing the
# gradient of the loss with respect to each intermediate hidden state.
# We do this BEFORE training (random init) and AFTER training.
#
# Method: register a hook on the hidden state at each time step
# to capture ∂L/∂h_t during backward.

class GradTrackingRNN(nn.Module):
    """Wrapper that exposes per-step hidden states so we can hook gradients."""
    def __init__(self, rnn_type, hidden_size, n_classes=6):
        super().__init__()
        self.rnn_type   = rnn_type
        self.hidden_size= hidden_size
        rnn_cls = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}[rnn_type]
        # Use our custom cells unrolled manually so we can hook each step
        self.rnn_cell   = rnn_cls(input_size=9, hidden_size=hidden_size,
                                  batch_first=True)
        self.head       = nn.Linear(hidden_size, n_classes)

    def forward(self, x):
        # Manual unroll so we can attach hooks to each h_t
        # x : (B, T, 9)
        B, T, _ = x.shape
        h = torch.zeros(1, B, self.hidden_size, device=x.device)
        c = torch.zeros(1, B, self.hidden_size, device=x.device)
        self.hidden_states = []

        for t in range(T):
            x_t = x[:, t:t+1, :]   # (B, 1, 9)
            if self.rnn_type == 'LSTM':
                out, (h, c) = self.rnn_cell(x_t, (h, c))
            else:
                out, h      = self.rnn_cell(x_t, h)
            # h: (1, B, hidden) – detach for storage but keep grad through the
            # original computation graph
            hs = h.squeeze(0)        # (B, hidden)
            hs.retain_grad()         # keep gradient for non-leaf tensor
            self.hidden_states.append(hs)

        last = self.hidden_states[-1]
        return self.head(last)       # (B, 6)


def compute_grad_norms(rnn_type, T=100, trained=False, seed=0):
    """Return per-step gradient norm array of length T."""
    set_seed(seed)
    model = GradTrackingRNN(rnn_type, HIDDEN_SIZE).to(DEVICE)

    if trained:
        # Train briefly (10 epochs) to get a 'trained' state
        set_seed(seed)
        X_tr2, y_tr2, X_val2, y_val2, m2, s2 = make_val_split(
            X_train_full, y_train_full, subj_train, seed)
        ds  = TensorDataset(torch.tensor(X_tr2),
                            torch.tensor(y_tr2, dtype=torch.long))
        dl  = DataLoader(ds, batch_size=128, shuffle=True)
        opt = torch.optim.Adam(model.parameters(), lr=LR)
        crit= nn.CrossEntropyLoss()
        for _ in range(10):
            model.train()
            for Xb, yb in dl:
                opt.zero_grad()
                crit(model(Xb.to(DEVICE)), yb.to(DEVICE)).backward()
                opt.step()

    # One forward/backward pass on a batch from UCI HAR (T=128→crop to T=100)
    set_seed(seed)
    X_tr2, y_tr2, _, _, m2, s2 = make_val_split(
        X_train_full, y_train_full, subj_train, seed)
    # Use T time steps
    Xb = torch.tensor(X_tr2[:128, :T, :]).to(DEVICE)
    yb = torch.tensor(y_tr2[:128], dtype=torch.long).to(DEVICE)

    model.eval()
    logits = model(Xb)
    loss   = nn.CrossEntropyLoss()(logits, yb)
    loss.backward()

    grad_norms = []
    for hs in model.hidden_states:
        if hs.grad is not None:
            grad_norms.append(hs.grad.norm(dim=1).mean().item())
        else:
            grad_norms.append(0.0)
    return np.array(grad_norms)

print("Computing gradient norms (before and after training)…")
T_GRAD = 100
grad_data = {}
for rnn_type in ['RNN', 'LSTM', 'GRU']:
    for trained in [False, True]:
        key = (rnn_type, trained)
        grad_data[key] = compute_grad_norms(rnn_type, T=T_GRAD, trained=trained)
        print(f"  {rnn_type} {'trained' if trained else 'untrained'}: done")

# ── Figure 5: gradient norm vs time step ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, rnn_type in zip(axes, ['RNN', 'LSTM', 'GRU']):
    steps = np.arange(1, T_GRAD + 1)
    ax.semilogy(steps, grad_data[(rnn_type, False)],
                color='tab:grey',  lw=1.5, ls='--', label='Before training')
    ax.semilogy(steps, grad_data[(rnn_type, True)],
                color=colors[rnn_type], lw=2,   label='After training')
    ax.set_title(rnn_type)
    ax.set_xlabel('Time step t')
    ax.legend(fontsize=8)
axes[0].set_ylabel('‖∂L/∂hₜ‖ (log scale)')
fig.suptitle('Figure 5: Gradient norm vs time step at T=100', fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig5_gradient_norms.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 5 saved.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 18 — Figure 6: Hidden-size sweep on HAR (Task 5b)         ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# We sweep hidden sizes {16, 32, 64, 128} for LSTM and GRU on HAR,
# using seed=0 only (to keep runtime reasonable).

HIDDEN_SWEEP = [16, 32, 64, 128]
print("Running hidden-size sweep (LSTM & GRU, seed=0)…")

sweep_results = {}  # {(rnn_type, h): result_dict}
for rnn_type in ['LSTM', 'GRU']:
    for h_size in HIDDEN_SWEEP:
        print(f"  {rnn_type} h={h_size} …", end='', flush=True)
        res = train_har(rnn_type, seed=0, hidden_size=h_size)
        sweep_results[(rnn_type, h_size)] = res
        print(f" acc={res['test_acc']:.4f}  params={res['params']:,}")

# ── Figure 6: accuracy & params vs hidden size ───────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for rnn_type, color in [('LSTM', 'tab:blue'), ('GRU', 'tab:green')]:
    accs   = [sweep_results[(rnn_type, h)]['test_acc']  for h in HIDDEN_SWEEP]
    params = [sweep_results[(rnn_type, h)]['params']    for h in HIDDEN_SWEEP]
    ax1.plot(HIDDEN_SWEEP, accs,   marker='o', color=color, label=rnn_type)
    ax2.plot(HIDDEN_SWEEP, params, marker='s', color=color, label=rnn_type, ls='--')

ax1.set_xlabel('Hidden size')
ax1.set_ylabel('Test accuracy')
ax1.set_title('Accuracy vs hidden size')
ax1.set_xticks(HIDDEN_SWEEP)
ax1.legend()

ax2.set_xlabel('Hidden size')
ax2.set_ylabel('Parameter count')
ax2.set_title('Parameters vs hidden size')
ax2.set_xticks(HIDDEN_SWEEP)
ax2.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
ax2.legend()

fig.suptitle('Figure 6: Hidden-size sweep – LSTM and GRU on HAR', fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig6_hidden_size_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 6 saved.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 19 — Figure 7: Task 5(c) ablation (forget-gate bias)      ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# VARIANT: last digit 7 (Odd) → LSTM forget-gate bias 0 vs 1 at T=200.
#
# The forget gate bias is at index h … 2h in the bias vector b_ih.
# When initialised to 1, the forget gate starts near σ(1) ≈ 0.73,
# which helps the LSTM remember earlier information at the start of
# training – often beneficial for long sequences.
#
# We compare both settings with 3 seeds each at T=200.

class LSTMForgetBiasModel(nn.Module):
    """LSTM adding-problem model with controllable forget-gate bias init."""
    def __init__(self, hidden_size=HIDDEN_SIZE, forget_bias=0.0):
        super().__init__()
        self.lstm = nn.LSTM(input_size=2, hidden_size=hidden_size,
                            batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

        # Set forget-gate bias (indices h … 2h in b_ih / b_hh)
        with torch.no_grad():
            h = hidden_size
            self.lstm.bias_ih_l0[h:2*h].fill_(forget_bias)
            self.lstm.bias_hh_l0[h:2*h].fill_(forget_bias)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)


def train_forget_bias(forget_bias, seed, T=200):
    set_seed(seed)
    x_tr,  y_tr  = make_adding(10000, T, seed=0)
    x_val, y_val = make_adding( 1000, T, seed=1)
    x_te,  y_te  = make_adding( 1000, T, seed=2)

    def make_loader(x, y, shuffle):
        ds = TensorDataset(torch.tensor(x), torch.tensor(y))
        return DataLoader(ds, batch_size=128, shuffle=shuffle)

    train_loader = make_loader(x_tr,  y_tr,  True)
    val_loader   = make_loader(x_val, y_val, False)
    test_loader  = make_loader(x_te,  y_te,  False)

    model  = LSTMForgetBiasModel(forget_bias=forget_bias).to(DEVICE)
    opt    = torch.optim.Adam(model.parameters(), lr=LR)
    crit   = nn.MSELoss()
    val_mses, train_mses = [], []

    for epoch in range(N_EPOCHS_ADD):
        model.train()
        for Xb, yb in train_loader:
            opt.zero_grad()
            crit(model(Xb.to(DEVICE)), yb.to(DEVICE)).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            t_m = np.mean([crit(model(Xb.to(DEVICE)), yb.to(DEVICE)).item()
                           for Xb, yb in train_loader])
            v_m = np.mean([crit(model(Xb.to(DEVICE)), yb.to(DEVICE)).item()
                           for Xb, yb in val_loader])
        train_mses.append(t_m)
        val_mses.append(v_m)

    test_mse = np.mean([crit(model(Xb.to(DEVICE)), yb.to(DEVICE)).item()
                        for Xb, yb in test_loader])
    return {'test_mse': test_mse, 'val_mses': val_mses, 'train_mses': train_mses}

print("Running Task 5(c) ablation: forget-gate bias 0 vs 1 at T=200…")
ablation_results = {0.0: [], 1.0: []}
for bias in [0.0, 1.0]:
    for seed in SEEDS:
        print(f"  Forget-bias={bias} seed={seed} …", end='', flush=True)
        res = train_forget_bias(bias, seed)
        ablation_results[bias].append(res)
        solved = "✓" if res['test_mse'] < 0.05 else "✗"
        print(f" MSE={res['test_mse']:.4f} {solved}")

# ── Figure 7: ablation learning curves ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
style_map = {0.0: ('tab:orange', 'Forget bias = 0'),
             1.0: ('tab:purple', 'Forget bias = 1')}

for ax, bias in zip(axes, [0.0, 1.0]):
    color, label = style_map[bias]
    for s_idx, res in enumerate(ablation_results[bias]):
        ep = range(1, len(res['val_mses']) + 1)
        ax.semilogy(ep, res['train_mses'], color=color, alpha=0.3, lw=1)
        ax.semilogy(ep, res['val_mses'],   color=color, alpha=0.9, lw=1.5,
                    ls='--', label=f'Val seed {SEEDS[s_idx]}')
    ax.axhline(BASELINE_MSE, color='grey', ls=':', lw=1.5, label='Baseline')
    ax.set_title(label)
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=7)

axes[0].set_ylabel('MSE (log scale)')
fig.suptitle('Figure 7: Task 5(c) ablation – LSTM forget-gate bias at T=200',
             fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig7_ablation_forget_bias.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 7 saved.")

# Summary
for bias in [0.0, 1.0]:
    mses   = [r['test_mse'] for r in ablation_results[bias]]
    solved = sum(1 for m in mses if m < 0.05)
    print(f"Forget-bias={bias}: MSE={np.mean(mses):.4f}±{np.std(mses):.4f}  "
          f"Solved {solved}/3")


## Save results.csv

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 20 — Build and save results.csv (Problem 1 rows)          ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# One row per run. Columns that do not apply are left empty (NaN).

result_rows = []

# HAR results
for rnn_type in ['RNN', 'LSTM', 'GRU']:
    for seed_idx, seed in enumerate(SEEDS):
        res = har_results[rnn_type][seed_idx]
        result_rows.append({
            'problem':       1,
            'model':         rnn_type,
            'seed':          seed,
            'seq_len':       128,
            'test_acc':      round(res['test_acc'],   4),
            'macro_f1':      round(res['macro_f1'],   4),
            'test_mse':      '',
            'params':        res['params'],
            'sec_per_epoch': round(res['sec_per_epoch'], 3),
            'val_loss_nats': '',
            'bits_per_char': '',
            'notes':         'HAR classification',
        })

# Adding-problem results
for T_val in T_VALUES:
    for rnn_type in ['RNN', 'LSTM', 'GRU']:
        for seed_idx, seed in enumerate(SEEDS):
            res = adding_results[(rnn_type, T_val)][seed_idx]
            result_rows.append({
                'problem':       1,
                'model':         rnn_type,
                'seed':          seed,
                'seq_len':       T_val,
                'test_acc':      '',
                'macro_f1':      '',
                'test_mse':      round(res['test_mse'],      4),
                'params':        '',
                'sec_per_epoch': round(res['sec_per_epoch'], 3),
                'val_loss_nats': '',
                'bits_per_char': '',
                'notes':         'Adding problem',
            })

# Ablation results
for bias in [0.0, 1.0]:
    for seed_idx, seed in enumerate(SEEDS):
        res = ablation_results[bias][seed_idx]
        result_rows.append({
            'problem':       1,
            'model':         'LSTM',
            'seed':          seed,
            'seq_len':       200,
            'test_acc':      '',
            'macro_f1':      '',
            'test_mse':      round(res['test_mse'], 4),
            'params':        '',
            'sec_per_epoch': '',
            'val_loss_nats': '',
            'bits_per_char': '',
            'notes':         f'Ablation forget_bias={bias}',
        })

results_df = pd.DataFrame(result_rows, columns=[
    'problem','model','seed','seq_len','test_acc','macro_f1',
    'test_mse','params','sec_per_epoch','val_loss_nats','bits_per_char','notes'
])

# Save to the parent folder (will be merged with Problem 2 results later)
results_df.to_csv('../results_p1.csv', index=False)
print("Problem 1 results saved to results_p1.csv")
print(f"Total rows: {len(results_df)}")
print(results_df.head(6).to_string(index=False))
